In [1]:
import pandas as pd
import sys
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from lazypredict.Supervised import LazyClassifier
import tabulate

from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils.helpers import Helpers

In [2]:
helper = Helpers()
properties = helper.load_properties()

try:
    feature_engineered_file_name = properties['LOCAL']['feature_engineered_file_name']
    RANDOM_STATE = properties['models']['random_state']
except KeyError as ke:
    raise KeyError(f"Missing key in properties file: {str(ke)}") from ke

In [3]:
# Apply global settings
helper.set_global_settings()

In [4]:
file_path = helper.root_dir / "datasets" / feature_engineered_file_name

df = pd.read_csv(file_path)
df.head()

,person_age,is_female,person_education,person_income,person_home_ownership,loan_amount,loan_intent,loan_interest_rate,credit_score,previous_loan_defaults,loan_status
0,-1.41,1,3,0.13,1,1.87,2,1.62,-1.38,0,1
1,-1.89,1,0,-2.82,3,-2.33,3,0.11,-2.14,1,0
2,-0.28,1,0,-2.80,2,-0.54,1,0.67,-0.04,0,1
3,-0.98,1,2,0.35,1,1.87,1,1.39,0.89,0,1
4,-0.61,0,3,-0.04,1,1.87,1,1.10,-0.98,0,1


In [5]:
df.shape

(42531, 11)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42531 entries, 0 to 42530
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   person_age              42531 non-null  float64
 1   is_female               42531 non-null  int64  
 2   person_education        42531 non-null  int64  
 3   person_income           42531 non-null  float64
 4   person_home_ownership   42531 non-null  int64  
 5   loan_amount             42531 non-null  float64
 6   loan_intent             42531 non-null  int64  
 7   loan_interest_rate      42531 non-null  float64
 8   credit_score            42531 non-null  float64
 9   previous_loan_defaults  42531 non-null  int64  
 10  loan_status             42531 non-null  int64  
dtypes: float64(5), int64(6)
memory usage: 3.6 MB


In [7]:
target_variable = 'loan_status'
X = df.drop(columns=[target_variable])
y = df[target_variable]

# Split the data into training (70%), validation (20%), and test (10%) sets
X_train, X_rem, y_train, y_rem = train_test_split(
    X, 
    y, 
    train_size=0.7, 
    stratify=y,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_rem, 
    y_rem, 
    train_size=2/3, 
    stratify=y_rem,
    random_state=RANDOM_STATE
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Validation set size: {X_val.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Training set size: 29771 samples
Validation set size: 8506 samples
Test set size: 4254 samples


In [8]:
columns_to_check = ['person_home_ownership', 'loan_intent']

for col in columns_to_check:
    freq_train = X_train[col].value_counts(normalize=True) * 100
    freq_val = X_val[col].value_counts(normalize=True) * 100

    dist = pd.concat(
        [freq_train, freq_val], 
        axis=1,
        keys=['Train(%)', 'Validation(%)']
    ).fillna(0).round(2)

    print(f'Category distribution for {col}:\n')
    print(dist.to_markdown(), '\n')

Category distribution for person_home_ownership:

|   person_home_ownership |   Train(%) |   Validation(%) |
|------------------------:|-----------:|----------------:|
|                       1 |      52.58 |           53.07 |
|                       2 |      40.7  |           40.12 |
|                       3 |       6.48 |            6.55 |
|                       0 |       0.25 |            0.26 | 

Category distribution for loan_intent:

|   loan_intent |   Train(%) |   Validation(%) |
|--------------:|-----------:|----------------:|
|             3 |      20.95 |           19.76 |
|             1 |      19.12 |           18.89 |
|             0 |      17.28 |           17.75 |
|             2 |      16.44 |           17.22 |
|             5 |      15.7  |           16.67 |
|             4 |      10.52 |            9.7  | 



In [9]:
freq_y_train = y_train.value_counts(normalize=True) * 100
freq_y_val = y_val.value_counts(normalize=True) * 100

dist = pd.concat(
    [freq_y_train, freq_y_val], 
    axis=1,
    keys=['Train(%)', 'Validation(%)']
).fillna(0).round(2)

print(f'Category distribution for {target_variable} (Target Variable):\n')
print(dist.to_markdown(), '\n')

Category distribution for loan_status (Target Variable):

|   loan_status |   Train(%) |   Validation(%) |
|--------------:|-----------:|----------------:|
|             0 |      77.74 |           77.73 |
|             1 |      22.26 |           22.27 | 



In [10]:
clf = LazyClassifier(
    predictions=True, 
    random_state=RANDOM_STATE
    )

models, predictions = clf.fit(X_train, X_val, y_train, y_val)
models.sort_values(by=['F1 Score', 'Balanced Accuracy', 'ROC AUC', 'Time Taken'], ascending=False)

  0%|          | 0/32 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 6628, number of negative: 23143
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000841 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1021
[LightGBM] [Info] Number of data points in the train set: 29771, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.222633 -> initscore=-1.250389
[LightGBM] [Info] Start training from score -1.250389


,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
XGBClassifier,0.93,0.88,0.88,0.93,0.64
LGBMClassifier,0.93,0.88,0.88,0.93,5.36
RandomForestClassifier,0.92,0.87,0.87,0.92,3.37
ExtraTreesClassifier,0.92,0.87,0.87,0.92,2.22
BaggingClassifier,0.92,0.86,0.86,0.92,0.64
SVC,0.91,0.86,0.86,0.91,12.96
AdaBoostClassifier,0.90,0.85,0.85,0.90,1.65
DecisionTreeClassifier,0.90,0.85,0.85,0.90,0.11
LogisticRegression,0.89,0.84,0.84,0.89,0.13


Top 5 models:

1. `XGBClassifier`

2. `LGBMClassifier`

3. `RandomForestClassifier`

4. `ExtraTreesClassifier`

5. `BaggingClassifier`

In [12]:
models_of_interest = [
    'XGBClassifier', 'LGBMClassifier', 'RandomForestClassifier', 'ExtraTreesClassifier', 'BaggingClassifier'
]

for model in models_of_interest:
    print('\t\t',model,'\n')
    print(classification_report(y_val, predictions[model]),'\n')

		 XGBClassifier 

              precision    recall  f1-score   support

           0       0.94      0.97      0.96      6612
           1       0.88      0.79      0.83      1894

    accuracy                           0.93      8506
   macro avg       0.91      0.88      0.89      8506
weighted avg       0.93      0.93      0.93      8506
 

		 LGBMClassifier 

              precision    recall  f1-score   support

           0       0.94      0.97      0.95      6612
           1       0.88      0.79      0.83      1894

    accuracy                           0.93      8506
   macro avg       0.91      0.88      0.89      8506
weighted avg       0.93      0.93      0.93      8506
 

		 RandomForestClassifier 

              precision    recall  f1-score   support

           0       0.94      0.97      0.95      6612
           1       0.87      0.78      0.82      1894

    accuracy                           0.92      8506
   macro avg       0.90      0.87      0.89      8506
wei